# Regional Obligations Profiling — SQL
Owner: Joy | Bronze: `pondolens.dev_bronze.regional_expenditure_obligations`
Status: working draft; run queries in order in Databricks. No Bronze or persistent Silver writes. All derived objects are session temporary views.

**BP:** How well does government spending align with regional needs and development indicators?
**Flow:** Allotments → Obligations → Disbursements.

| Question | Contribution and limitation |
|---|---|
| AP1: Where does the money go across sectors and regions? | Verified regional obligations show geographic distribution. Department/SPF labels are not sectors; use a verified sector crosswalk before sector analysis. |
| AP2: Where do budget execution gaps appear? | Obligations alone cannot measure utilization or payment gaps. Require comparable allotments and disbursements with matching year, scope, unit and reporting cutoff. GAA minus obligations is not unused allotment. |
| AP3: How does regional spending compare with population and economic conditions? | Join verified regional obligations to same-year, compatible-boundary population and GRDP. Per-person obligations are descriptive; they do not prove adequate spending or outcomes. |

**Inspection findings from the submitted file:** no saved query outputs were present. Its narrative reports 4,046 rows (1,785 + 2,261), conflicting year fields, two source layouts and stale duplicate checks. These are historical claims to recheck, not current results. The earlier 1,785-row baseline must not be a hard failure rule. `price_basis="Actual obligations"` describes a measure, not a price basis. Zero amounts require source review; do not delete automatically.


## Inspect Columns
Inspect the actual schema before accepting candidate field mappings below. JSON extraction allows absent optional fields without unresolved-column errors, but missing mappings remain missing and must be corrected. Inspect every source layout; do not silently merge conflicting fields.

In [0]:
%sql
DESCRIBE TABLE pondolens.dev_bronze.regional_expenditure_obligations;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW ro_raw AS
SELECT to_json(struct(*), map('ignoreNullFields','false')) AS raw_json
FROM pondolens.dev_bronze.regional_expenditure_obligations;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW ro_fields AS
SELECT raw_json,
get_json_object(raw_json, '$.fiscal_year') AS fiscal_year,
get_json_object(raw_json, '$.year') AS year,
get_json_object(raw_json, '$.region') AS region,
get_json_object(raw_json, '$.regions') AS regions,
get_json_object(raw_json, '$.amount_obligated') AS amount_obligated,
get_json_object(raw_json, '$.expenditure_amount') AS expenditure_amount,
get_json_object(raw_json, '$.category') AS category,
get_json_object(raw_json, '$.particulars') AS particulars,
get_json_object(raw_json, '$.department') AS department,
get_json_object(raw_json, '$.department_name') AS department_name,
get_json_object(raw_json, '$.unit') AS unit,
get_json_object(raw_json, '$.unit_amount') AS unit_amount,
get_json_object(raw_json, '$.price_basis') AS price_basis,
get_json_object(raw_json, '$.source_file') AS source_file,
get_json_object(raw_json, '$._source_file') AS _source_file,
get_json_object(raw_json, '$.file_name') AS file_name,
get_json_object(raw_json, '$._rescued_data') AS _rescued_data,
get_json_object(raw_json, '$._ingested_at') AS _ingested_at
FROM ro_raw;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW ro_profile AS
WITH chosen AS (
 SELECT *,
 CASE WHEN nullif(trim(amount_obligated),'') IS NOT NULL THEN 'amount_obligated_layout'
      WHEN nullif(trim(expenditure_amount),'') IS NOT NULL THEN 'expenditure_amount_layout'
      ELSE 'unresolved_layout' END AS source_layout,
 coalesce(nullif(trim(fiscal_year),''),nullif(trim(year),'')) AS year_raw,
 coalesce(nullif(trim(region),''),nullif(trim(regions),'')) AS region_raw,
 coalesce(nullif(trim(amount_obligated),''),nullif(trim(expenditure_amount),'')) AS amount_raw,
 coalesce(nullif(trim(particulars),''),nullif(trim(department),''),nullif(trim(department_name),'')) AS entity_raw,
 coalesce(source_file,_source_file,file_name,'UNAVAILABLE') AS source_file_raw
 FROM ro_fields
)
SELECT *,
 CASE WHEN year_raw RLIKE '^[0-9]{4}([.]0+)?$' THEN try_cast(year_raw AS DECIMAL(6,1)) END AS year_numeric,
 try_cast(regexp_replace(amount_raw, ',', '') AS DECIMAL(38,2)) AS amount_numeric,
 lower(regexp_replace(trim(region_raw),'\\s+',' ')) AS region_key,
 lower(regexp_replace(trim(entity_raw),'\\s+',' ')) AS entity_key,
 lower(regexp_replace(trim(category),'\\s+',' ')) AS category_key,
 (nullif(trim(fiscal_year),'') IS NOT NULL AND nullif(trim(year),'') IS NOT NULL AND trim(fiscal_year) <> trim(year)) AS year_conflict,
 (nullif(trim(region),'') IS NOT NULL AND nullif(trim(regions),'') IS NOT NULL AND lower(trim(region)) <> lower(trim(regions))) AS region_conflict,
 (nullif(trim(amount_obligated),'') IS NOT NULL AND nullif(trim(expenditure_amount),'') IS NOT NULL AND trim(amount_obligated) <> trim(expenditure_amount)) AS amount_conflict
FROM chosen;

## Preview source records
Check both layouts independently. Raw JSON retains original values and ingestion metadata. Candidate entity mappings must be confirmed against the schema and original files.

In [0]:
%sql
SELECT source_layout, source_file_raw, count(*) AS rows FROM ro_profile GROUP BY ALL ORDER BY ALL;

In [0]:
%sql
SELECT * FROM ro_profile
QUALIFY row_number() OVER (PARTITION BY source_layout,source_file_raw ORDER BY raw_json) <= 10;

## 1. Year coverage
Target scope is FY2021–2025. Missing years stay missing. Conflicting fields are quarantined, not silently resolved.

In [0]:
%sql
SELECT source_layout, fiscal_year, year, year_numeric, count(*) AS rows
FROM ro_profile GROUP BY ALL ORDER BY source_layout,year_numeric;

In [0]:
%sql
WITH expected AS (SELECT explode(array(2021,2022,2023,2024,2025)) AS fiscal_year)
SELECT e.fiscal_year, count(p.year_numeric) AS records,
 CASE WHEN count(p.year_numeric)=0 THEN 'MISSING — investigate availability' ELSE 'PRESENT — verify source status' END AS coverage
FROM expected e LEFT JOIN ro_profile p ON e.fiscal_year=p.year_numeric
GROUP BY e.fiscal_year ORDER BY e.fiscal_year;

## 2. Missing and invalid values
Profile all Bronze fields, including absent values across layouts. Placeholders remain null candidates, never zero. Amount commas are removed for parsing only; verify source formatting and scale separately.

In [0]:
%sql
WITH fields AS (
 SELECT source_layout, explode(from_json(raw_json,'MAP<STRING,STRING>')) AS (column_name,value)
 FROM ro_profile
)
SELECT source_layout,column_name,count(*) AS rows,
 count_if(value IS NULL) AS nulls,
 count_if(value IS NOT NULL AND trim(value)='') AS blanks,
 count_if(upper(trim(value)) IN ('-','--','N/A','NA','...','NULL')) AS placeholders
FROM fields GROUP BY ALL ORDER BY source_layout,column_name;

In [0]:
%sql
SELECT source_layout,count(*) AS rows,
 count_if(year_numeric IS NULL OR year_numeric NOT BETWEEN 2021 AND 2025) AS invalid_or_missing_year,
 count_if(region_raw IS NULL) AS missing_region,
 count_if(entity_raw IS NULL) AS missing_entity_mapping,
 count_if(amount_raw IS NULL) AS missing_amount,
 count_if(amount_raw IS NOT NULL AND amount_numeric IS NULL) AS invalid_amount,
 count_if(amount_numeric<0) AS negatives_review,
 count_if(amount_numeric=0) AS zeros_review,
 count_if(year_conflict OR region_conflict OR amount_conflict) AS conflicting_fields,
 count_if(_rescued_data IS NOT NULL AND trim(_rescued_data) NOT IN ('','{}','null')) AS rescued_rows
FROM ro_profile GROUP BY source_layout;

In [0]:
%sql
SELECT * FROM ro_profile WHERE year_numeric IS NULL OR amount_numeric IS NULL
 OR year_conflict OR region_conflict OR amount_conflict OR amount_numeric<0
 OR (_rescued_data IS NOT NULL AND trim(_rescued_data) NOT IN ('','{}','null')) LIMIT 100;

## 3. Duplicate Records
Check exact payload duplicates ignoring underscore metadata and common source-file fields. Proposed business grain: source layout × fiscal year × region × category × entity. Confirm additional source dimensions before declaring repeated keys duplicates. No automatic deduplication.

In [0]:
%sql
WITH payload AS (
 SELECT to_json(map_filter(from_json(raw_json,'MAP<STRING,STRING>'),
  (k,v) -> NOT startswith(k,'_') AND k NOT IN ('source_file','file_name','file_path'))) AS business_payload
 FROM ro_raw
)
SELECT business_payload,count(*) AS occurrences,count(*)-1 AS excess_rows
FROM payload GROUP BY business_payload HAVING count(*)>1 ORDER BY occurrences DESC;

In [0]:
%sql
SELECT source_layout,year_numeric,region_key,category_key,entity_key,
 count(*) AS records,count(DISTINCT amount_numeric) AS distinct_amounts,
 collect_set(source_file_raw) AS files
FROM ro_profile GROUP BY ALL HAVING count(*)>1 ORDER BY records DESC;

## 4. Polished Totals
These are **diagnostic raw sums**, not approved spending totals: totals, subtotals, repeated ingestion and incompatible units may coexist. Do not publish them. Separate layouts and units; preserve decimal precision. “Actual obligations” must not be treated as a price basis.

In [0]:
%sql
SELECT source_layout,year_numeric,unit,unit_amount,price_basis,
 count(*) AS rows,count(amount_numeric) AS valid_amounts,
 min(amount_numeric) AS minimum,max(amount_numeric) AS maximum,
 sum(amount_numeric) AS diagnostic_raw_sum_not_for_publication,
 percentile_approx(amount_numeric,array(0.01,0.5,0.99)) AS percentiles
FROM ro_profile GROUP BY ALL ORDER BY source_layout,year_numeric;

### Verified classification configuration
Populate `ro_rules` only after source tick-out. One rule per layout × category × entity × region. `row_role`: DETAIL, REPORTED_TOTAL, SUBTOTAL or EXCLUDE. `geo_scope`: REGION, NATIONWIDE, CENTRAL_OFFICE or NATIONAL_TOTAL. `reconciliation_group` must identify one non-overlapping accounting scope; detail and its reported total share the same group. `unit_multiplier` converts to PHP (1 for pesos, 1000 for thousands, 1000000 for millions). `measure_status` must be ACTUAL_OBLIGATIONS for approved output. Empty rules intentionally produce no approved totals. Geographic rules must enumerate verified labels; no regex automatically certifies totals.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW ro_rules AS
SELECT CAST(NULL AS STRING) AS source_layout,CAST(NULL AS STRING) AS category_key,
 CAST(NULL AS STRING) AS entity_key,CAST(NULL AS STRING) AS region_key,
 CAST(NULL AS STRING) AS row_role,CAST(NULL AS STRING) AS geo_scope,
 CAST(NULL AS STRING) AS reconciliation_group,
 CAST(NULL AS DECIMAL(18,6)) AS unit_multiplier,
 CAST(NULL AS STRING) AS measure_status,
 CAST(NULL AS STRING) AS source_reference
WHERE false;

In [0]:
%sql
SELECT source_layout,category_key,entity_key,region_key,count(*) AS rules
FROM ro_rules GROUP BY ALL HAVING count(*)>1;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW ro_classified AS
SELECT p.*,r.row_role,r.geo_scope,r.reconciliation_group,r.unit_multiplier,
 r.measure_status,r.source_reference,
 p.amount_numeric*r.unit_multiplier AS amount_php
FROM ro_profile p LEFT JOIN ro_rules r
ON p.source_layout=r.source_layout AND p.category_key <=> r.category_key
 AND p.entity_key <=> r.entity_key AND p.region_key <=> r.region_key;

In [0]:
%sql
SELECT source_layout,year_numeric,row_role,geo_scope,count(*) AS rows,
 sum(amount_php) AS configured_sum_php
FROM ro_classified GROUP BY ALL ORDER BY ALL;

## 5. National total vs regional sum
For each verified, non-overlapping scope, compare **one** reported national total against REGION + NATIONWIDE + CENTRAL_OFFICE detail. Nationwide and Central Office are not regions; retain separately. More than one reported total makes the comparison invalid. Missing totals are UNAVAILABLE, not a pass. Do not hide small differences with an assumed tolerance; record exact differences and investigate source rounding. Historical context mentions a ₱1,000 FY2022 difference, which must be reverified.

In [0]:
%sql
WITH a AS (
 SELECT source_layout,year_numeric,reconciliation_group,
 count_if(row_role='REPORTED_TOTAL' AND geo_scope='NATIONAL_TOTAL') AS reported_total_rows,
 sum(CASE WHEN row_role='REPORTED_TOTAL' AND geo_scope='NATIONAL_TOTAL' THEN amount_php END) AS reported_national_php,
 sum(CASE WHEN row_role='DETAIL' AND geo_scope='REGION' THEN amount_php END) AS regional_php,
 sum(CASE WHEN row_role='DETAIL' AND geo_scope='NATIONWIDE' THEN amount_php END) AS nationwide_php,
 sum(CASE WHEN row_role='DETAIL' AND geo_scope='CENTRAL_OFFICE' THEN amount_php END) AS central_office_php,
 count_if(amount_php IS NULL) AS invalid_amount_rows
 FROM ro_classified WHERE row_role IN ('DETAIL','REPORTED_TOTAL')
 GROUP BY ALL
), b AS (
 SELECT *,regional_php+coalesce(nationwide_php,0)+coalesce(central_office_php,0) AS combined_detail_php FROM a
)
SELECT *, reported_national_php-combined_detail_php AS difference_php,
 CASE WHEN reported_total_rows<>1 OR invalid_amount_rows>0 OR combined_detail_php IS NULL THEN 'UNAVAILABLE / INVALID'
 WHEN reported_national_php=combined_detail_php THEN 'MATCH'
 ELSE 'DIFFERENCE — investigate source and coverage' END AS result
FROM b ORDER BY source_layout,year_numeric,reconciliation_group;

## 6. Other relevant checks — BP/AP readiness
Review label variants, geography coverage, source provenance, amount scale, measure and totals. Historical boundaries (including NIR) must match population and GRDP before joins. Never allocate Nationwide/Central Office to regions without an evidence-backed method.

AP1 sector mapping needs a verified department/SPF-to-sector crosswalk with unmapped entries reported. AP2 requires actual allotments and disbursements; do not assume GAA is allotment. AP3 requires unique region/year indicator keys, matching boundaries, positive population, explicit observation year and price basis. Do not carry forward missing indicators or mix nominal obligations with real GRDP without explanation. External table names in the original notebook were unverified, so cross-table joins are not enabled here.

In [0]:
%sql
SELECT region_key,collect_set(region_raw) AS original_labels,count(*) AS rows,
 collect_set(year_raw) AS years
FROM ro_profile GROUP BY region_key ORDER BY region_key;

In [0]:
%sql
SELECT source_layout,year_numeric,region_key,count(*) AS rows,
 count(DISTINCT entity_key) AS entities
FROM ro_profile GROUP BY ALL ORDER BY ALL;

In [0]:
%sql
SELECT source_layout,category_key,entity_key,region_key,count(*) AS rows
FROM ro_profile
WHERE entity_key RLIKE '(total|subtotal|grand total)' OR region_key RLIKE '(nation|central|philippines|total)'
GROUP BY ALL ORDER BY ALL;

### Original-source verification log
Tick beginning/middle/end samples in each source and each layout; check national totals and unusual amounts. Record: File | Original sheet/page/cell | Expected value and unit | Converted value and unit | MATCH/ISSUE | Notes. Text-pattern hits above are review candidates only. Query order is deterministic, not random sampling.

In [0]:
%sql
WITH ranked AS (
 SELECT *,row_number() OVER(PARTITION BY source_layout,source_file_raw ORDER BY raw_json) AS rn,
 count(*) OVER(PARTITION BY source_layout,source_file_raw) AS n FROM ro_profile
)
SELECT * FROM ranked WHERE rn IN (1,CAST(ceil(n/2.0) AS BIGINT),n)
ORDER BY source_layout,source_file_raw,rn;

## 7. Silver Preparation
PROPOSED: retain source fields and provenance; standardize types; resolve conflicting layouts; certify actual measure and PHP multiplier; classify row and geographic scope; maintain historical region IDs through a governed crosswalk. Keep zero values pending review. Investigate negatives rather than silently deleting. Quarantine unresolved records. Store reported totals separately from additive detail. Deduplication requires a verified key and deterministic source precedence.

The view below is a **review preview**, not a production Silver load. Its READY_FOR_REVIEW label is not publication approval: uniqueness, reconciliation, source tick-out and geography mapping must pass first.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW ro_silver_preview AS
SELECT CAST(year_numeric AS INT) AS fiscal_year,region_raw,region_key,
 category AS category_raw,category_key,entity_raw,entity_key,
 amount_raw,amount_numeric,unit AS unit_raw,unit_amount AS unit_amount_raw,
 unit_multiplier,amount_php,row_role,geo_scope,reconciliation_group,measure_status,
 source_layout,source_file_raw,source_reference,_ingested_at AS ingested_at_raw,raw_json,
 CASE WHEN year_numeric IS NULL OR year_numeric NOT BETWEEN 2021 AND 2025
 OR region_raw IS NULL OR entity_raw IS NULL OR amount_numeric IS NULL
 OR year_conflict OR region_conflict OR amount_conflict
 OR (_rescued_data IS NOT NULL AND trim(_rescued_data) NOT IN ('','{}','null'))
 OR row_role IS NULL OR geo_scope IS NULL OR reconciliation_group IS NULL
 OR unit_multiplier IS NULL OR unit_multiplier<=0 OR measure_status<>'ACTUAL_OBLIGATIONS'
 OR measure_status IS NULL OR source_reference IS NULL
 THEN 'QUARANTINE'
 WHEN amount_numeric<0 THEN 'NEGATIVE — SOURCE REVIEW'
 ELSE 'READY_FOR_REVIEW' END AS review_status
FROM ro_classified;

In [0]:
%sql
SELECT review_status,source_layout,count(*) AS rows FROM ro_silver_preview GROUP BY ALL ORDER BY ALL;

## 8. Silver Columns
| Proposed column | Type | Purpose |
|---|---|---|
| fiscal_year | INT | Verified observation year |
| region_id | STRING, pending crosswalk | Stable historical geography identifier; absent from preview until mapped |
| region_raw / region_key | STRING | Original label / normalized matching candidate |
| category_raw / category_key | STRING | Source classification; not automatically a sector |
| entity_raw / entity_key | STRING | Verified department/SPF or source entity grain |
| amount_raw / amount_numeric | STRING / DECIMAL(38,2) | Original / parsed source amount |
| unit_raw / unit_amount_raw | STRING | Original unit evidence |
| unit_multiplier / amount_php | DECIMAL | Verified conversion / PHP amount; derived precision inspected below |
| row_role / geo_scope | STRING | Detail vs control total; geography scope |
| reconciliation_group | STRING | Comparable non-overlapping accounting scope |
| measure_status | STRING | Verified ACTUAL_OBLIGATIONS |
| source_layout / source_file_raw / source_reference | STRING | Batch layout and source evidence |
| ingested_at_raw / raw_json | STRING | Preserved ingestion evidence and original record |
| review_status | STRING | Quarantine or source-review routing |

**Separate production outputs:** additive detail; reported-total controls; quarantined rows. Region IDs, source-year metadata, reporting cutoff, price basis and sector IDs are added only when verified. Do not infer them from “Actual obligations.”

**Run conclusion:** record live counts, invalid values, repeated keys, classification coverage, reconciliation differences, source evidence and remaining blockers. No current Databricks results are asserted in this notebook.


In [0]:
%sql
DESCRIBE ro_silver_preview;

In [0]:
%sql
SELECT * FROM ro_silver_preview ORDER BY source_layout,fiscal_year,region_key,entity_key LIMIT 50;

##Regions

In [0]:
SELECT DISTINCT region AS region_list
FROM pondolens.dev_bronze.regional_expenditure_obligations
WHERE region IS NOT NULL
ORDER BY region_list;